# 01 - Data Loading and Exploratory Data Analysis

Loads the Bank Customer Churn dataset and the CFPB banking-complaints subset, then produces Figures 4.1-4.3 used in the dissertation.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve().parent))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')

from code import config, data_loader, preprocess

bank = data_loader.load_bank_churn()
bank.head()

In [ ]:
bank = preprocess.clean_bank_churn(bank)
print(bank.shape)
print('Churn rate:', bank[config.TARGET_COL].mean())

In [ ]:
# CFPB - skip if file not present
try:
    cfpb = data_loader.load_cfpb_complaints()
    cfpb['narrative'] = cfpb['narrative'].apply(preprocess.clean_narrative)
    cfpb = data_loader.stratified_sample(cfpb, by='product', n=config.CFPB_SAMPLE_SIZE)
    cfpb.to_parquet(config.PROCESSED_DIR / 'cfpb_sample.parquet', index=False)
    print('CFPB sample shape:', cfpb.shape)
except FileNotFoundError as e:
    print('CFPB skipped:', e)

In [ ]:
# Regenerate figures 4.1-4.3 from real data when available
%run ../scripts/make_figures.py